# EY Data Challenge — v4: Benchmark-first

**Philosophy:** Start from what WORKS (benchmark=0.20), then add ONE thing at a time.

| Approach | Features | Change from benchmark |
|----------|----------|----------------------|
| **A** | swir22, NDMI, MNDWI, pet | Exact reproduction (sanity check) |
| **B** | same 4 | Train on 100% data instead of 70% |
| **C** | same 4 + month_sin/cos | Add seasonality |
| **D** | all 6 bands + pet + month | Add all raw Landsat bands |

**Key fixes vs v3:** No log transforms, StandardScaler, val NaN filled with val median, no spectral ratios.


## Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

## FILE PATHS — UPDATE THESE

In [ ]:
WATER_QUALITY_PATH = 'water_quality_training_dataset.csv'
LANDSAT_TRAIN_PATH = 'landsat_features_training.csv'
TERRA_TRAIN_PATH   = 'terraclimate_features_training.csv'
LANDSAT_VAL_PATH   = 'landsat_features_validation.csv'
TERRA_VAL_PATH     = 'terraclimate_features_validation.csv'
TEMPLATE_PATH      = 'submission_template.csv'
OUTPUT_PATH        = 'submission_v4.csv'

## 1. LOAD & MERGE (identical to benchmark)

In [ ]:
print("Loading data...")
wq = pd.read_csv(WATER_QUALITY_PATH)
landsat_train = pd.read_csv(LANDSAT_TRAIN_PATH)
terra_train = pd.read_csv(TERRA_TRAIN_PATH)
landsat_val = pd.read_csv(LANDSAT_VAL_PATH)
terra_val = pd.read_csv(TERRA_VAL_PATH)
template = pd.read_csv(TEMPLATE_PATH)

# Merge training (same as benchmark)
train = pd.concat([wq, landsat_train, terra_train], axis=1)
train = train.loc[:, ~train.columns.duplicated()]
train = train.fillna(train.median(numeric_only=True))  # same as benchmark

# Build validation (same as benchmark)
val = pd.DataFrame({
    'Longitude': landsat_val['Longitude'].values,
    'Latitude': landsat_val['Latitude'].values,
    'Sample Date': landsat_val['Sample Date'].values,
    'nir': landsat_val['nir'].values,
    'green': landsat_val['green'].values,
    'swir16': landsat_val['swir16'].values,
    'swir22': landsat_val['swir22'].values,
    'NDMI': landsat_val['NDMI'].values,
    'MNDWI': landsat_val['MNDWI'].values,
    'pet': terra_val['pet'].values,
})
val = val.fillna(val.median(numeric_only=True))  # SAME AS BENCHMARK: val median

TARGETS = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']

print(f"Training: {train.shape}, Validation: {val.shape}")

## 2. APPROACH A: EXACT BENCHMARK REPRODUCTION (sanity check)

In [ ]:
print("\n" + "="*70)
print("APPROACH A: Exact benchmark reproduction")
print("="*70)

BASE_FEATURES = ['swir22', 'NDMI', 'MNDWI', 'pet']

X_train_base = train[BASE_FEATURES]
X_val_base = val[BASE_FEATURES]

preds_A = {}
for target in TARGETS:
    y = train[target]
    
    # 70/30 split (same as benchmark: random_state=42)
    from sklearn.model_selection import train_test_split
    X_tr, X_te, y_tr, y_te = train_test_split(X_train_base, y, test_size=0.3, random_state=42)
    
    # Scale (same as benchmark)
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)
    
    # Train (same as benchmark)
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_tr_s, y_tr)
    
    # Evaluate on held-out 30%
    r2_test = r2_score(y_te, model.predict(X_te_s))
    
    # Predict validation
    X_val_s = scaler.transform(X_val_base)
    preds_A[target] = model.predict(X_val_s)
    
    short = target.split()[0] if 'Dissolved' not in target else 'DRP'
    print(f"  {short:15s}  30% test R²={r2_test:.4f}  val_pred: min={preds_A[target].min():.1f} max={preds_A[target].max():.1f} mean={preds_A[target].mean():.1f}")

## 3. APPROACH B: Train on ALL data (only change from benchmark)

In [ ]:
print("\n" + "="*70)
print("APPROACH B: Same features + RF, but train on 100% data")
print("="*70)

preds_B = {}
for target in TARGETS:
    y = train[target]
    
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_train_base)
    
    model = RandomForestRegressor(n_estimators=200, random_state=42)
    model.fit(X_tr_s, y)
    
    X_val_s = scaler.transform(X_val_base)
    preds_B[target] = model.predict(X_val_s)
    
    short = target.split()[0] if 'Dissolved' not in target else 'DRP'
    print(f"  {short:15s}  val_pred: min={preds_B[target].min():.1f} max={preds_B[target].max():.1f} mean={preds_B[target].mean():.1f}")

## 4. APPROACH C: Add time features only (safe addition)

In [ ]:
print("\n" + "="*70)
print("APPROACH C: Benchmark features + time features + RF")
print("="*70)

# Add time features to both train and val
def add_time_features(df):
    df = df.copy()
    dates = pd.to_datetime(df['Sample Date'], format='mixed', dayfirst=True)
    df['month_sin'] = np.sin(2 * np.pi * dates.dt.month / 12)
    df['month_cos'] = np.cos(2 * np.pi * dates.dt.month / 12)
    return df

train_t = add_time_features(train)
val_t = add_time_features(val)

FEATURES_C = BASE_FEATURES + ['month_sin', 'month_cos']

X_train_C = train_t[FEATURES_C]
X_val_C = val_t[FEATURES_C]

preds_C = {}
for target in TARGETS:
    y = train[target]
    
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_train_C)
    
    model = RandomForestRegressor(n_estimators=200, random_state=42)
    model.fit(X_tr_s, y)
    
    X_val_s = scaler.transform(X_val_C)
    preds_C[target] = model.predict(X_val_s)
    
    short = target.split()[0] if 'Dissolved' not in target else 'DRP'
    print(f"  {short:15s}  val_pred: min={preds_C[target].min():.1f} max={preds_C[target].max():.1f} mean={preds_C[target].mean():.1f}")

## 5. APPROACH D: All Landsat bands + time + RF (more features but no ratios)

In [ ]:
print("\n" + "="*70)
print("APPROACH D: All raw Landsat bands + time + RF")
print("="*70)

FEATURES_D = ['nir', 'green', 'swir16', 'swir22', 'NDMI', 'MNDWI', 'pet', 'month_sin', 'month_cos']

X_train_D = train_t[FEATURES_D].fillna(train_t[FEATURES_D].median())
X_val_D = val_t[FEATURES_D].fillna(val_t[FEATURES_D].median())

preds_D = {}
for target in TARGETS:
    y = train[target]
    
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_train_D)
    
    model = RandomForestRegressor(n_estimators=200, random_state=42)
    model.fit(X_tr_s, y)
    
    X_val_s = scaler.transform(X_val_D)
    preds_D[target] = model.predict(X_val_s)
    
    short = target.split()[0] if 'Dissolved' not in target else 'DRP'
    print(f"  {short:15s}  val_pred: min={preds_D[target].min():.1f} max={preds_D[target].max():.1f} mean={preds_D[target].mean():.1f}")

## 6. SPATIAL CV for all approaches (to estimate which is best BEFORE submitting)

In [ ]:
print("\n" + "="*70)
print("SPATIAL CV COMPARISON (estimating leaderboard performance)")
print("="*70)

coords = train[['Latitude', 'Longitude']].values
clusters = KMeans(n_clusters=12, random_state=42, n_init=10).fit_predict(coords)
gkf = GroupKFold(n_splits=5)

def spatial_cv_rf(X, y, clusters):
    """Spatial CV with RF + StandardScaler (matching benchmark approach)."""
    oof = np.zeros(len(X))
    fold_r2s = []
    
    for train_idx, val_idx in gkf.split(X, y, clusters):
        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X.iloc[train_idx])
        X_val_s = scaler.transform(X.iloc[val_idx])
        
        model = RandomForestRegressor(n_estimators=200, random_state=42)
        model.fit(X_tr_s, y.iloc[train_idx])
        
        p = model.predict(X_val_s)
        oof[val_idx] = p
        fold_r2s.append(r2_score(y.iloc[val_idx], p))
    
    return r2_score(y, oof), fold_r2s

approaches = {
    'A (bench 4 feat)': X_train_base,
    'C (+time)':        X_train_C,
    'D (+all bands)':   X_train_D,
}

for name, X in approaches.items():
    print(f"\n--- {name} ---")
    for target in TARGETS:
        y = train[target]
        oof_r2, fold_r2s = spatial_cv_rf(X, y, clusters)
        short = target.split()[0] if 'Dissolved' not in target else 'DRP'
        print(f"  {short:15s}  OOF R²={oof_r2:.4f}  worst_fold={min(fold_r2s):.4f}  folds={[f'{r:.3f}' for r in fold_r2s]}")

## 7. GENERATE SUBMISSIONS (one per approach — pick the best from spatial CV)

In [ ]:
print("\n" + "="*70)
print("SAVING SUBMISSIONS")
print("="*70)

all_preds = {
    'A': preds_A,
    'B': preds_B,
    'C': preds_C,
    'D': preds_D,
}

for label, preds in all_preds.items():
    sub = template.copy()
    sub['Total Alkalinity'] = preds['Total Alkalinity']
    sub['Electrical Conductance'] = preds['Electrical Conductance']
    sub['Dissolved Reactive Phosphorus'] = preds['Dissolved Reactive Phosphorus']
    
    fname = f'submission_v4_{label}.csv'
    sub.to_csv(fname, index=False)
    print(f"  Saved {fname}")
    
    # Show prediction comparison
    for target in TARGETS:
        short = target.split()[0] if 'Dissolved' not in target else 'DRP'
        vals = preds[target]
        print(f"    {short}: mean={vals.mean():.1f} std={vals.std():.1f} range=[{vals.min():.1f}, {vals.max():.1f}]")

# Also save the best approach as the main submission
# Approach A should match benchmark. Pick the best from spatial CV for main.
print(f"\n✓ Main submission saved as: {OUTPUT_PATH}")
print("Pick the approach with best spatial CV worst-fold R² and submit that file.")
print("Start with approach A to verify you match the 0.20 benchmark.")